# Problem Set: Choosing — and pricing — the formalism for an ED ontology with Claude
### Chapter 10 — Rough, Temporal, and Fuzzy Modelling

**Topic coverage:** Keet §10.1 (temporal: Allen's interval algebra, composition, path consistency), §10.2 (vagueness: fuzzy membership, t-norms, alpha-cuts; granularity: rough sets, lower/upper approximation); course notebooks 01–03 of this chapter. Tools: DSPy + GEPA, LangChain agents, the Chapter 10 interval, fuzzy and rough-set engine.

**Level:** Graduate (practice-first). Combines conceptual analysis with building, measuring and optimising a real Claude-backed agent.

**Total points:** 100 (Part A 28 / Part B 28 / Part C 24 / Part D 20); 53 pts are auto-graded by the check cells, the rest are marked by hand against the rubric. **Estimated effort:** 10–12 hours.

**API budget:** ≈ \$5–12 **estimated** (not measured) on `claude-opus-5` for a full run: ≈ 200 DSPy calls at roughly \$0.02–0.05 each (the two GEPA runs are the largest lines) plus 8 agent episodes at roughly \$0.10–0.25 each. Re-running unchanged cells is free — DSPy caches identical requests (except in C3, which disables the cache on purpose). Every model call in this notebook is a billed request to Anthropic; the cost lines the problems ask for are part of the answer.

## The brief

The Emergency Department of **Northfield General Hospital** is building a
patient-flow and sepsis early-warning ontology. Its clinical-informatics lead,
Dr. Amara Okafor, has a backlog of 24 requirements from clinicians, managers and
the pharmacy committee. Each is imperfect in exactly one way: it relates things
that happen **over time**, it uses a predicate with **no sharp boundary** ("a high
temperature"), it has to be answered from data **too coarse** to separate the
cases, or it is plain **crisp**. Each needs a different formalism, and each
formalism has a price: deciding consistency of a general Allen network is
NP-complete, while fuzzy membership and rough approximation are polynomial.

You are the ontology engineer. Your brief:

1. **Justify the labels with the engine.** Before anyone automates the choice,
   show on the ED's own data — the triage vitals, the coded triage log, the
   sepsis-bundle protocol — what each formalism buys and what it costs.
2. **Build a grader and a Claude advisor** that, given a requirement, names the
   formalism **and** its reasoning cost. An advisor that picks right but cannot
   price the choice will happily hand the ED an NP-complete reasoning problem it
   did not need. Optimise it with GEPA on `train`, select on `dev`, report on a
   held-out `test`, with cost and run-to-run noise.
3. **Ship a review agent** that looks at the data attached to each requirement
   and runs the engine before it recommends anything.
4. **Price the search.** Model constraint propagation as an MDP and work out when
   proving (in)consistency is worth what it costs — and what an honest reward for
   "consistent" has to look like.

The backlog (24 items, 6 per formalism, split 8 / 8 / 8 by item — 2 per formalism
per split), the ED's data and the protocol are provided in `ch10_agentic.py`; every
gold label is re-derived from its attached data by `A.validate_gold()`.

## Submission requirements

1. **This notebook, executed top to bottom**, with every `TODO` implemented and every *Your answer* cell written.
2. **The artefacts it saves** under `course/artifacts/` (optimised instructions, run logs) — they are evidence, not by-products.
3. **Honest reporting.** A number without its split, its sample size and its cost earns no credit. Report the runs you did, including the ones that did not improve anything.

The check cells are the marking scheme for the auto-graded points; do not edit them. The final self-test prints your auto-graded subtotal.

> Solutions: `04_solutions.ipynb` — do not open it until you have attempted every problem.

In [ ]:
import sys, json, logging
from pathlib import Path

# Make the course root and this chapter importable no matter where Jupyter started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break
sys.path.insert(0, str(here))

import oe_course
from oe_course import config
from oe_course.grading import Grader

print(json.dumps(oe_course.describe_environment(), indent=1))
config.require_credentials()          # this problem set calls the live Anthropic API
logging.getLogger("dspy").setLevel(logging.WARNING)
GRADER = Grader('Chapter 10 — Rough, Temporal, and Fuzzy Modelling — Choosing — and pricing — the formalism for an ED ontology with Claude')

import itertools, random, re, statistics as st, time
import dspy
import numpy as np
import pandas as pd
import ch10_toolkit as ch10
import ch10_agentic as A
from oe_course import evaluation as ev, llm, mdp, optimize as opt, agents

GOLD = A.validate_gold()          # every gold label re-derived from its witness data
train, dev, test = A.build_dataset("train"), A.build_dataset("dev"), A.build_dataset("test")
print(f"train {len(train)} / dev {len(dev)} / test {len(test)}")
pd.DataFrame([{"id": r.id, "split": r.split, "formalism": r.formalism,
               "cost": A.EXPRESSIVITY_COST[r.formalism], "requirement": r.text}
              for r in A.REQUIREMENTS])

---

## Part A — What each formalism buys, on the ED's own data

Everything in this part is deterministic and free: no model calls. The point is to be
able to *defend* a label for each kind of requirement with the Chapter 10 engine before
you ask Claude to make the choice — and to know what the choice costs.

### A1. A threshold is confidently wrong at the boundary (9 pts)

The ED's sepsis screen currently reads "high temperature" as `temp >= 38.0` °C.
Clinicians defend thresholds anywhere between 37.8 and 38.5. The ED's own vague
predicates are in `A.VAGUE_PREDICATES` (trapezoids) and `A.degree(name, value)`; the
evening's triage vitals are in `A.VITALS` (`visit -> (temp, heart_rate)`).

(a) Build `a1`, one row per visit, with columns `visit`, `temp`, `hr`, `fever`
(`A.degree("high_fever", temp)`), `crisp_38` (`temp >= 38.0`), `contested` (the crisp
decision differs between the thresholds 37.8 and 38.5), `sepsis_min` and
`sepsis_product` (the fuzzy conjunction of `fever` and `A.degree("tachycardia", hr)`
under `ch10.fuzzy_and` with the `min` and the `product` t-norm).

(b) Implement `threshold_for_alpha(alpha)`: the temperature `T` such that the
alpha-cut `{t : fever(t) >= alpha}` is exactly `{t : t >= T}`, for `0 < alpha <= 1`.
Read the ramp off `A.VAGUE_PREDICATES["high_fever"]` — do not hard-code it.

(c) In writing: which visits is the crisp rule *confidently wrong* about, and which
visit does *any* fever threshold miss? What does an alpha-cut buy over the crisp rule
if it also ends in a threshold? Which t-norm would you use for "high fever **and**
fast heart rate", and why?

In [ ]:
a1 = None                     # TODO (a)

def threshold_for_alpha(alpha: float) -> float:
    # TODO (b)
    raise NotImplementedError

In [ ]:
# Marking scheme for A1 -- do not edit.
with GRADER.check('A1', points=5):
    assert isinstance(a1, pd.DataFrame) and len(a1) == len(A.VITALS)
    assert {"visit", "temp", "hr", "fever", "crisp_38", "contested",
            "sepsis_min", "sepsis_product"} <= set(a1.columns)
    for r in a1.itertuples():
        t, h = A.VITALS[r.visit]
        assert abs(r.fever - A.degree("high_fever", t)) < 1e-9
        assert bool(r.crisp_38) == (t >= 38.0)
        assert bool(r.contested) == ((t >= 37.8) != (t >= 38.5))
        f, g = A.degree("high_fever", t), A.degree("tachycardia", h)
        assert abs(r.sepsis_min - min(f, g)) < 1e-3 and abs(r.sepsis_product - f * g) < 1e-3
    contested = a1[a1.contested.astype(bool)]
    assert len(contested) > 0 and ((contested.fever > 0) & (contested.fever < 1)).all(), \
        "every contested visit sits on the fuzzy ramp"
    grid = [round(36.0 + 0.05 * i, 2) for i in range(100)]
    for alpha in (0.25, 0.5, 0.75, 1.0):
        T = threshold_for_alpha(alpha)
        assert [t for t in grid if A.degree("high_fever", t) >= alpha] == \
               [t for t in grid if t >= T - 1e-9], f"alpha={alpha}"

**Your answer:**

*(Write your answer here.)*

### A2. What the triage screen cannot decide (8 pts)

The old triage screen kept four coded fields (`A.TRIAGE_LOG`); the discharge records
say which visits went on to intensive care (`A.TRIAGE_TARGETS["icu"]`).

(a) Build `a2` with one row for **every non-empty subset** of the recorded attributes
and columns `attributes` (a sorted tuple), `n_attributes`, `granules` (number of
indiscernibility classes), `lower`, `upper` (sizes of the approximations), `boundary`
(the list of visits) and `accuracy`.

(b) Set `best_attributes` to the smallest subset reaching the maximum accuracy (ties:
fewest attributes, then the alphabetically first tuple).

(c) In writing: what can the ED say about ICU admission from the triage screen — what
is certain, what is possible, what is undecidable? Is any recorded field redundant?
What would you ask the ED to start recording, and why is this a rough-set problem
rather than a fuzzy one?

In [ ]:
a2 = None                 # TODO (a)
best_attributes = None    # TODO (b)

raise NotImplementedError

In [ ]:
# Marking scheme for A2 -- do not edit.
with GRADER.check('A2', points=5):
    system, target = A.TRIAGE_LOG, A.TRIAGE_TARGETS["icu"]
    assert isinstance(a2, pd.DataFrame) and len(a2) == 2 ** len(system.attributes()) - 1
    for r in a2.itertuples():
        attrs = tuple(r.attributes)
        assert attrs == tuple(sorted(attrs)) and r.n_attributes == len(attrs)
        assert abs(r.accuracy - system.accuracy(target, attrs)) < 1e-9
        assert list(r.boundary) == system.boundary(target, attrs)
        assert r.granules == len(system.indiscernibility(attrs))
    acc = {tuple(r.attributes): r.accuracy for r in a2.itertuples()}
    for s in acc:
        for t in acc:
            if set(s) < set(t):
                assert acc[s] <= acc[t] + 1e-9, "a finer partition can never lose accuracy"
    top = max(acc.values())
    expected = min((k for k, v in acc.items() if v == top), key=lambda k: (len(k), k))
    assert tuple(best_attributes) == expected

**Your answer:**

*(Write your answer here.)*

### A3. A protocol, an amendment, and the price of deciding (11 pts)

`A.SEPSIS_PROTOCOL` is the ED's sepsis bundle as Allen constraints between episodes
(Triage, Cultures, Antibiotics, Lactate, the one-hour Bundle window). The pharmacy
committee proposes `A.PROTOCOL_AMENDMENT`: start antibiotics *at* triage.

(a) Run `ch10.path_consistent` on the protocol and on protocol + amendment. Print what
the protocol **implies** for the Triage–Antibiotics pair, and set `amendment_conflict`
to the `empty_pair` reported for the amended network.

(b) Path consistency is sound but incomplete. Implement a complete decision procedure
`satisfiable(constraints, nodes=None) -> (bool, explored)`: build the network, run path
consistency, and if every label is a singleton stop (path consistency decides atomic
Allen networks); otherwise pick an undecided pair with the fewest candidate relations,
try each relation in turn on a **copy**, and recurse. `explored` counts the networks
visited (the root included). Do not mutate the caller's constraints.

(c) Show that `A.PC_BLIND_SPOT` is accepted by path consistency and rejected by your
procedure.

(d) Measure the price: for `n` in 3…8 intervals and label sizes `k` in (3, 5, 7), draw
10 random complete networks each (`random.Random(1000 * n + k)`, each label a
`rng.sample(ch10.ALLEN_RELATIONS, k)`), and build `a3` with columns `n`, `k`,
`satisfiable_rate`, `mean_explored`, `max_explored`, `pc_blind` (networks path
consistency accepted but `satisfiable` rejected). Keep the cell under 10 s.

(e) In writing: the price list says temporal reasoning costs **high**, yet path
consistency is only cubic. Reconcile the two using (c) and (d). What should the ED do
about the amendment, and what does a `"consistent": true` from a path-consistency tool
actually license an agent to say?

In [ ]:
amendment_conflict = None    # TODO (a)

def satisfiable(constraints, nodes=None):
    # TODO (b): backtracking over relations + path consistency; return (bool, explored)
    raise NotImplementedError

a3 = None                    # TODO (d)

In [ ]:
# Marking scheme for A3 -- do not edit.
with GRADER.check('A3', points=5):
    _net = lambda c: ch10.Network.complete(tuple(sorted({x for p in c for x in p})),
                                           {k: set(v) for k, v in c.items()})
    _amended = {**A.SEPSIS_PROTOCOL, **A.PROTOCOL_AMENDMENT}
    assert amendment_conflict == ch10.path_consistent(_net(_amended))["empty_pair"]
    assert set(amendment_conflict.split("-")) == {"Antibiotics", "Triage"}
    _before = {k: set(v) for k, v in A.SEPSIS_PROTOCOL.items()}
    assert satisfiable(A.SEPSIS_PROTOCOL)[0] is True
    assert {k: set(v) for k, v in A.SEPSIS_PROTOCOL.items()} == _before, "do not mutate the input"
    assert satisfiable(_amended)[0] is False
    assert ch10.path_consistent(_net(A.PC_BLIND_SPOT))["consistent"]
    assert satisfiable(A.PC_BLIND_SPOT)[0] is False, "the blind spot has no model"
    for req in A.REQUIREMENTS:          # every temporal witness has a concrete timeline
        if req.witness[0] == "allen":
            assert satisfiable(req.witness[1])[0] is True, req.id
    _rng = random.Random(7)             # atomic networks read off concrete intervals
    _iv = {f"I{i}": tuple(sorted(_rng.sample(range(12), 2))) for i in range(5)}
    _atomic = {(a, b): {ch10.relation_between(_iv[a], _iv[b])}
               for a, b in itertools.combinations(sorted(_iv), 2)}
    assert satisfiable(_atomic) == (True, 1)
    assert satisfiable({("A", "B"): {"b"}, ("B", "C"): {"b"}, ("A", "C"): {"bi"}})[0] is False
    assert isinstance(a3, pd.DataFrame) and set(a3["n"]) == set(range(3, 9)) and set(a3["k"]) == {3, 5, 7}
    assert {"satisfiable_rate", "mean_explored", "max_explored", "pc_blind"} <= set(a3.columns)
    assert (a3.mean_explored >= 1).all() and a3.satisfiable_rate.between(0, 1).all()

**Your answer:**

*(Write your answer here.)*

---

## Part B — Grade the choice and its price, then ask Claude

The advisor answers two things per requirement — the formalism and its reasoning cost —
and is graded on both. The guidelines a scorer can report are in `A.FORMALISM_RULEBOOK`:

In [ ]:
for rule in A.FORMALISM_RULEBOOK:
    print(f"{rule.id:34s} {rule.description[:90]}")
print()
print("price list:", A.EXPRESSIVITY_COST)

### B1. A scorer that charges for expressivity (14 pts)

Implement `formalism_scorer(gold, pred) -> ev.ScoreReport`, where `gold` is a dataset
row (`gold.gold_formalism`, `gold.gold_cost`) and `pred` has `formalism` and `cost`.
First **normalise** each field: `str`, lower-case, strip whitespace and the characters
`` "'`*.: `` from both ends. Then:

| outcome | score | `violated` |
|---|---|---|
| formalism not one of `A.FORMALISMS` | 0.0 | `["answer-with-a-label"]` |
| right formalism, right cost | 1.0 | `[]` |
| right formalism, cost wrong (or not a cost level) | 0.6 | `["price-the-expressivity"]` |
| wrong formalism, cost = the price of **its own** choice | 0.2 | `[RULE_FOR[gold]]` |
| wrong formalism, cost not the price of its own choice | 0.0 | `[RULE_FOR[gold], "price-the-expressivity"]` |

`RULE_FOR` maps each formalism to its rule (`temporal-for-interval-relations`, …).
`notes` must state what was produced, what was intended and why
(`A.FORMALISM_WHY[gold]`); add a note mentioning the word **threshold** when a fuzzy
requirement was answered *crisp* (the §10.2 failure), and one mentioning
**over-engineer** when a crisp requirement was answered with anything else (paying
reasoning cost for nothing). Those notes are what GEPA will read.

Why grade the cost against the gold formalism's price, and why give 0.2 to a wrong
choice that is at least *priced consistently*? Answer in one paragraph.

In [ ]:
RULE_FOR = {}    # TODO: formalism -> rule id

def normalise(value) -> str:
    # TODO
    raise NotImplementedError

def formalism_scorer(gold, pred) -> ev.ScoreReport:
    # TODO: the five outcomes from the table, with diagnostic notes
    raise NotImplementedError

In [ ]:
# Marking scheme for B1 -- do not edit.
with GRADER.check('B1', points=11):
    fuzzy_gold = next(ex for ex in dev if ex.id == "fast-heart-rate")
    crisp_gold = next(ex for ex in dev if ex.id == "ward-capacity")
    P = lambda f, c: dspy.Prediction(formalism=f, cost=c)
    cases = [
        (fuzzy_gold, "fuzzy", "low", 1.0, []),
        (fuzzy_gold, " Fuzzy", "Low.", 1.0, []),
        (fuzzy_gold, "**fuzzy**", "`low`", 1.0, []),
        (fuzzy_gold, "fuzzy", "none", 0.6, ["price-the-expressivity"]),
        (fuzzy_gold, "fuzzy", "cheap", 0.6, ["price-the-expressivity"]),
        (fuzzy_gold, "crisp", "none", 0.2, ["fuzzy-for-vague-predicates"]),
        (fuzzy_gold, "rough", "low", 0.2, ["fuzzy-for-vague-predicates"]),
        (fuzzy_gold, "crisp", "high", 0.0, ["fuzzy-for-vague-predicates", "price-the-expressivity"]),
        (fuzzy_gold, "fuzzy logic", "low", 0.0, ["answer-with-a-label"]),
        (fuzzy_gold, "", "", 0.0, ["answer-with-a-label"]),
        (fuzzy_gold, None, None, 0.0, ["answer-with-a-label"]),
        (crisp_gold, "crisp", "none", 1.0, []),
        (crisp_gold, "temporal", "high", 0.2, ["crisp-when-boundaries-are-sharp"]),
    ]
    for gold, f, c, score, violated in cases:
        r = formalism_scorer(gold, P(f, c))
        assert abs(r.score - score) < 1e-9 and r.violated == violated, (gold.id, f, c, r.score, r.violated)
        assert all(v in A.FORMALISM_RULEBOOK for v in r.violated)
    assert set(RULE_FOR) == set(A.FORMALISMS) and all(v in A.FORMALISM_RULEBOOK for v in RULE_FOR.values())
    r = formalism_scorer(fuzzy_gold, P("crisp", "none"))
    assert any("threshold" in n.lower() for n in r.notes)
    assert any("fuzzy" in n for n in r.notes), "say what was intended"
    r = formalism_scorer(crisp_gold, P("temporal", "high"))
    assert any("over-engineer" in n.lower() for n in r.notes)

**Your answer:**

*(Write your answer here.)*

### B2. The advisor as a DSPy program (6 pts)

Write a signature `FormalismChoice` with input `requirement` and outputs `formalism`,
`cost` and `justification`, and a factory `FormalismAdvisor(instruction)` returning a
`dspy.Module` with a single `dspy.Predict` whose instruction is `instruction`. The field
descriptions are part of the prompt (manual marks): say which labels are allowed. Then
configure Claude and run the program once on `train[0]`.

In [ ]:
BASELINE_INSTRUCTION = "You are a knowledge engineer. Choose a representation for the requirement, and price it."

# TODO: class FormalismChoice(dspy.Signature): ...
# TODO: def FormalismAdvisor(instruction=BASELINE_INSTRUCTION): ...

lm = llm.configure_dspy()
smoke = None     # TODO: FormalismAdvisor()(**train[0].inputs())

raise NotImplementedError

In [ ]:
# Marking scheme for B2 -- do not edit.
with GRADER.check('B2', points=3):
    assert set(FormalismChoice.input_fields) == {"requirement"}
    assert {"formalism", "cost", "justification"} <= set(FormalismChoice.output_fields)
    program = FormalismAdvisor("custom instruction")
    assert len(list(program.named_predictors())) == 1
    assert opt.instruction_of(program) == "custom instruction"
    assert isinstance(smoke.formalism, str) and smoke.formalism.strip()

### B3. Baseline on the development split, with a confusion matrix (8 pts)

Run the baseline advisor on `dev` inside `llm.meter(lm)` (store the cost in
`baseline_dev_cost`) and collect `b3` with columns `id`, `gold`, `predicted`
(normalised), `cost` (normalised), `score` and `violated` from your scorer. Print the
mean and `pd.crosstab(b3.gold, b3.predicted)`.

Then write the **error analysis**: which confusions occur (crisp ↔ fuzzy on
"under 16" / "at most 24"? rough ↔ fuzzy on "coarse" data?), which items lose only the
price, and whether each error is a *choice* error or a *pricing* error — they call for
different fixes.

In [ ]:
# TODO: b3 = ..., baseline_dev_cost = ...

raise NotImplementedError

In [ ]:
# Marking scheme for B3 -- do not edit.
with GRADER.check('B3', points=3):
    assert isinstance(b3, pd.DataFrame) and len(b3) == len(dev) == 8
    assert list(b3["id"]) == [ex.id for ex in dev]
    for r, ex in zip(b3.itertuples(), dev):
        again = formalism_scorer(ex, dspy.Prediction(formalism=r.predicted, cost=r.cost))
        assert abs(again.score - r.score) < 1e-9, "score must be the scorer's verdict on the row"
    assert {"calls", "usd"} <= set(baseline_dev_cost)

**Your answer:**

*(Write your answer here.)*

---

## Part C — Optimise with GEPA — and report it honestly

The rules: **optimise on `train`, select on `dev`, report on `test`**, and put a cost and
a noise estimate next to every number.

### C1. A budgeted GEPA run with a held-out report (10 pts)

1. Build the GEPA feedback metric from your scorer and `A.FORMALISM_RULEBOOK`, and a
   separate reflection LM (`llm.reflection_lm()`).
2. Run `opt.run_gepa` on `train` with `valset=dev` and `max_metric_calls=GEPA_BUDGET`
   inside `llm.meter(lm, reflect)`; store the cost in `gepa_cost` and the program in
   `tuned`.
3. Compare baseline and tuned on **`test`** with `opt.compare` (store as `c1`) and
   print `c1.report()`.
4. Save the tuned instruction to `config.artifacts_dir() /
   "ch10_formalism_advisor_instruction.txt"`.

In writing: read the instruction diff. Did GEPA write down the price list? Which
guidelines did it *not* learn, and was each of them ever violated on `train`? (A rule
for behaviour the model already has is never learned — that is not a failure.)

In [ ]:
GEPA_BUDGET = 60
# TODO: gepa_metric, reflect, tuned (inside llm.meter -> gepa_cost), c1, save the instruction
instruction_path = config.artifacts_dir() / "ch10_formalism_advisor_instruction.txt"

raise NotImplementedError

In [ ]:
# Marking scheme for C1 -- do not edit.
with GRADER.check('C1', points=4):
    ids = lambda rows: {ex.id for ex in rows}
    assert not (ids(test) & (ids(train) | ids(dev))), "the test split must be untouched"
    assert {r["item"] for r in c1.after["rows"]} == ids(test), "report on the test split"
    assert c1.before["n"] == c1.after["n"] == len(test)
    assert instruction_path.is_file()
    assert instruction_path.read_text(encoding="utf-8") == opt.instruction_of(tuned)
    assert {"calls", "usd"} <= set(gepa_cost) and GEPA_BUDGET <= 100

**Your answer:**

*(Write your answer here.)*

### C2. Score the choice only — and see what it learns about cost (8 pts)

Implement `choice_only_scorer(gold, pred)`: 1.0 for the right formalism, else 0.0, with
the same `answer-with-a-label` / `RULE_FOR[gold]` violations as B1 and no cost
component. Run a second GEPA (`max_metric_calls=40`) against it — `blind` — inside
`llm.meter` (`blind_cost`).

Build `c2` with one row per program (`"baseline"`, `"gepa_full"`, `"gepa_choice_only"`)
and columns `test_full` (mean under `formalism_scorer`), `test_choice_only` (mean under
`choice_only_scorer`), `priced_rate` (fraction of test items **without** a
`price-the-expressivity` violation under the full scorer) and `optimisation_usd`.
Evaluating the same program twice costs nothing extra: DSPy serves the second pass from
its cache.

In writing: what did the choice-only optimiser learn about cost, and what does this say
about what a metric has to contain?

In [ ]:
def choice_only_scorer(gold, pred) -> ev.ScoreReport:
    # TODO
    raise NotImplementedError

c2 = None     # TODO

In [ ]:
# Marking scheme for C2 -- do not edit.
with GRADER.check('C2', points=4):
    _g = next(ex for ex in dev if ex.id == "sepsis-audit")
    _P = lambda f, c="": dspy.Prediction(formalism=f, cost=c)
    assert choice_only_scorer(_g, _P("Rough", "high")).score == 1.0
    assert choice_only_scorer(_g, _P("rough")).violated == []
    r = choice_only_scorer(_g, _P("fuzzy", "low"))
    assert r.score == 0.0 and r.violated == ["rough-for-indiscernible-data"]
    assert choice_only_scorer(_g, _P("rough sets")).violated == ["answer-with-a-label"]
    assert isinstance(c2, pd.DataFrame)
    assert set(c2["program"]) == {"baseline", "gepa_full", "gepa_choice_only"}
    assert {"test_full", "test_choice_only", "priced_rate", "optimisation_usd"} <= set(c2.columns)
    assert c2.priced_rate.between(0, 1).all() and c2.test_full.between(0, 1).all()
    assert c2.set_index("program").loc["baseline", "optimisation_usd"] == 0

**Your answer:**

*(Write your answer here.)*

### C3. Is the difference bigger than the noise? (6 pts)

With caching **disabled** (`fresh = llm.dspy_lm(cache=False)` and
`with dspy.context(lm=fresh): ...`), run the baseline and `tuned` on `test` three times
each under `formalism_scorer`; store the means in
`runs = {"baseline": [..3..], "gepa": [..3..]}`, the cost in `c3_cost`, and set
`verdict_c3` to `"significant"` if
`|mean_gepa - mean_baseline| > 2 * max(sd_gepa, sd_baseline)`, else
`"not significant"`. In writing: what does this say about C1, and what would a credible
evaluation need?

In [ ]:
fresh = llm.dspy_lm(cache=False)
runs = {"baseline": [], "gepa": []}
# TODO: fill runs (inside llm.meter(fresh) as c3_cost), then compute verdict_c3
verdict_c3 = None

raise NotImplementedError

In [ ]:
# Marking scheme for C3 -- do not edit.
with GRADER.check('C3', points=3):
    assert set(runs) == {"baseline", "gepa"} and all(len(v) == 3 for v in runs.values())
    assert all(0.0 <= x <= 1.0 for v in runs.values() for x in v)
    _m = {k: st.mean(v) for k, v in runs.items()}
    _s = {k: st.stdev(v) for k, v in runs.items()}
    _expected = ("significant" if abs(_m["gepa"] - _m["baseline"]) > 2 * max(_s.values())
                 else "not significant")
    assert verdict_c3 == _expected
    assert {"calls", "usd"} <= set(c3_cost)

**Your answer:**

*(Write your answer here.)*

---

## Part D — A review agent that runs the engine, and the price of the search

The DSPy advisor sees only the English. In practice a requirement arrives with data — a
measure and its observed values, a table and a target set, or proposed interval
constraints — and the right formalism is the one the **data** demonstrates. The review
agent's tools (`A.build_review_tools`) wrap the engine; your job is the agent, its output
contract and its evaluation.

In [ ]:
ws_demo = A.ReviewWorkspace()
demo = {t.name: t for t in A.build_review_tools(ws_demo)}
for name, t in demo.items():
    print(f"{name:28s} {t.description.splitlines()[0]}")
print()
print(demo["get_requirement"].invoke({"requirement_id": "icu-from-triage"}))
print(demo["rough_report"].invoke({"table": "triage_log", "attributes": "acuity,arrival",
                                   "target_set": "icu"}))
print("trajectory:", ws_demo.log.names())

### D1. Build and evaluate the requirements-review agent (9 pts)

1. Write `REVIEW_PROMPT`. It must make the agent fetch the requirement's data, run the
   engine tool that tests its hypothesis (`fuzzy_degree` on the observed values,
   `rough_report` on the table — an empty boundary means *crisp* —,
   `check_temporal_consistency` on the constraints), price the choice with
   `expressivity_cost`, and end with a JSON object
   `{"formalism": ..., "cost": ..., "evidence": [...]}`.
2. Write `parse_review(text) -> dict` returning `{"formalism": ..., "cost": ...}`
   (normalised with your `normalise`) from the **last** JSON object in the text that
   has a `formalism` key — it may sit in a code fence or after prose — or
   `{"formalism": "unparseable", "cost": "unparseable"}`.
3. For every **test** requirement, build a fresh `A.ReviewWorkspace()`, its tools, an
   agent (`agents.build_agent(ws, system_prompt=..., tools=...)`) and run it with the
   task `"Review requirement <id>."`. Collect `d1` with columns `id`, `gold`,
   `predicted`, `cost`, `correct` (formalism and cost both right), `tool_calls`,
   `grounded` (the log contains the engine tool for the *predicted* formalism:
   fuzzy → `fuzzy_degree`, rough/crisp → `rough_report`, temporal →
   `check_temporal_consistency`) and `usd_estimate` (`llm.chat_usage(run.messages)`).

In writing: report accuracy with its n and cost per requirement, compare with the
text-only advisor on the same eight items (C1's `c1.after`), and examine every error and
every ungrounded answer. Is the agent worth ~10× the cost per item?

In [ ]:
REVIEW_PROMPT = """TODO"""

def parse_review(text: str) -> dict:
    # TODO
    raise NotImplementedError

ENGINE_TOOL = {}   # TODO: formalism -> the tool that demonstrates it
d1 = None          # TODO: run the agent on every test requirement

In [ ]:
# Marking scheme for D1 -- do not edit.
with GRADER.check('D1', points=4):
    assert parse_review('{"formalism": "rough", "cost": "low", "evidence": []}') == {"formalism": "rough", "cost": "low"}
    assert parse_review('Checked.\n```json\n{"formalism": "Temporal", "cost": "HIGH", "evidence": ["x"]}\n```')["formalism"] == "temporal"
    assert parse_review('{"degree": 0.5} then {"formalism": "fuzzy", "cost": "low"}')["cost"] == "low"
    assert parse_review("I think it is crisp.")["formalism"] == "unparseable"
    assert ENGINE_TOOL == {"fuzzy": "fuzzy_degree", "rough": "rough_report", "crisp": "rough_report",
                           "temporal": "check_temporal_consistency"}
    assert isinstance(d1, pd.DataFrame) and list(d1["id"]) == [ex.id for ex in test]
    assert {"id", "gold", "predicted", "cost", "correct", "tool_calls", "grounded",
            "usd_estimate"} <= set(d1.columns)
    _gold_cost = {ex.id: ex.gold_cost for ex in test}
    assert (d1["correct"] == ((d1["predicted"] == d1["gold"]) &
                              (d1["cost"] == d1["id"].map(_gold_cost)))).all()
    assert d1["grounded"].map(lambda v: isinstance(v, (bool, np.bool_))).all()

**Your answer:**

*(Write your answer here.)*

### D2. The price of the search: propagation as an MDP (11 pts)

`A.PropagationMDP` models deciding a three-interval network: propagate through A, B or
C (each costs `cost`) or declare a verdict. A **justified** correct verdict earns +1; a
wrong or unjustified one costs `wrong_penalty`. Declaring *inconsistent* is justified
only once a label has emptied; with `require_check=True`, declaring *consistent* needs
at least one propagation.

The amendment from A3, restricted to Triage (A), Cultures (B), Antibiotics (C), is
provided below as `amended_kwargs`; the protocol's consistent chain as `chain_kwargs`.

1. Write `plan(M) -> (actions, value)`: solve `M` with `mdp.value_iteration`, roll out
   the greedy policy from the initial state, and return the action list and
   `V*(s0)`. Set `k` to the number of propagations in the optimal plan for the
   amendment at `cost=0.05`.
2. Derive the propagation cost `c*` above which the optimal agent stops proving
   inconsistency, as a function of `k` and the penalty `p`; implement
   `threshold(k, p)`. Verify it: for `p` in `[1.0, 2.0]`, sweep `cost` over
   `np.linspace(0.05, 3.0, 30)` and record in `d2` (columns `penalty`, `cost`,
   `proves`) whether the optimal plan propagates and ends in `declare:inconsistent`.
3. On the consistent chain, compare `require_check=False` and `True`: set
   `loose_plan, loose_value, strict_plan, strict_value`.

In writing: interpret `c*` as a solver time budget; explain why the loose reward is a
design flaw rather than a fact about temporal reasoning, and why even
`require_check=True` does not make "consistent" a proof (A3's blind spot). What reward
would?

In [ ]:
amended_kwargs = dict(ab={"b", "m"}, bc={"b", "m"}, ac={"eq", "s", "si"}, consistent=False)
chain_kwargs = dict(ab={"b"}, bc={"b"}, ac={"b"}, consistent=True)
M_demo = A.PropagationMDP(**amended_kwargs)
print("states:", len(M_demo.states()), " actions at s0:", M_demo.actions(M_demo.initial_state()))

In [ ]:
def plan(M):
    # TODO (1): (actions, V*(s0))
    raise NotImplementedError

k = None                      # TODO (1)

def threshold(k: int, p: float) -> float:
    # TODO (2)
    raise NotImplementedError

d2 = None                     # TODO (2)
loose_plan = loose_value = strict_plan = strict_value = None   # TODO (3)

In [ ]:
# Marking scheme for D2 -- do not edit.
with GRADER.check('D2', points=6):
    assert k == 1
    assert abs(threshold(1, 1.0) - 2.0) < 1e-9 and abs(threshold(4, 2.0) - 0.75) < 1e-9
    assert isinstance(d2, pd.DataFrame) and set(d2["penalty"]) == {1.0, 2.0}
    for p, grp in d2.groupby("penalty"):
        c_star = threshold(k, p)
        clear = grp[(grp["cost"] - c_star).abs() > 0.06]
        assert (clear.proves == (clear["cost"] < c_star)).all(), f"sweep disagrees with c* at p={p}"
    _M = A.PropagationMDP(**amended_kwargs)
    _acts, _v = plan(_M)
    assert _acts[-1] == "declare:inconsistent" and abs(_v - (1 - _M.cost)) < 1e-9
    assert loose_plan == ["declare:consistent"] and abs(loose_value - 1.0) < 1e-9
    assert any(a.startswith("propagate") for a in strict_plan) and strict_plan[-1] == "declare:consistent"
    assert abs(strict_value - (1 - 0.05)) < 1e-9 and strict_value < loose_value

**Your answer:**

*(Write your answer here.)*

---

## Grading rubric

| Part | Problem | Auto | Manual | Pts |
|---|---|---|---|---|
| A | A1 A threshold is confidently wrong at the boundary | 5 | 4 | 9 |
| A | A2 What the triage screen cannot decide | 5 | 3 | 8 |
| A | A3 A protocol, an amendment, and the price of deciding | 5 | 6 | 11 |
| B | B1 A scorer that charges for expressivity | 11 | 3 | 14 |
| B | B2 The advisor as a DSPy program | 3 | 3 | 6 |
| B | B3 Baseline on the development split, with a confusion matrix | 3 | 5 | 8 |
| C | C1 A budgeted GEPA run with a held-out report | 4 | 6 | 10 |
| C | C2 Score the choice only — and see what it learns about cost | 4 | 4 | 8 |
| C | C3 Is the difference bigger than the noise? | 3 | 3 | 6 |
| D | D1 Build and evaluate the requirements-review agent | 4 | 5 | 9 |
| D | D2 The price of the search: propagation as an MDP | 6 | 5 | 11 |
| **Total** | | | | **100** |

*Manual* points are for the written answers and for the quality of the engineering evidence (splits, sample sizes, cost, honest negative results).

---

## Self-test

Re-run this cell after finishing. It lists every auto-graded problem and whether its check passed.

In [ ]:
AUTO_GRADED = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3', 'D1', 'D2']
_ = GRADER.summary(AUTO_GRADED)